In [ ]:
import pandas as pd
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC

In [ ]:
df= pd.read_csv('loan_data.csv')
df.head()

In [ ]:
x=df.drop(columns=['loan_status'])
y=df.loan_status

In [ ]:
xtrain,xtest,ytrain,ytest=train_test_split(x,y,train_size=0.8,random_state=42)

In [ ]:
obj_cols=x.select_dtypes(include='object').columns
obj_cols

In [ ]:
xtrain[obj_cols].nunique()

In [ ]:
df['person_education'].unique()

In [ ]:
preprocessing=ColumnTransformer(
    transformers=[
        ('oh_encoder', OneHotEncoder(sparse_output=False, handle_unknown='ignore'),
          obj_cols.drop('person_education')),
        ('ord_encoder', OrdinalEncoder(categories=[['Master', 'High School', 'Bachelor', 'Associate', 'Doctorate']], handle_unknown='use_encoded_value', unknown_value=-1),['person_education'])
    ],
    remainder='passthrough'
)

In [ ]:
main_pipeline=Pipeline(
    steps=[
        ('preprocessing',preprocessing),
        ('model', SVC())
    ]
)

In [ ]:
grid_search_cv= GridSearchCV(
    estimator=main_pipeline,
    param_grid={
        'model__kernel':['linear', 'poly', 'rbf', 'sigmoid'],
        'model__C':[0.01,0.1,1.0,10
]
    },
    cv=3,
    verbose=10,
    n_jobs=-1
)
grid_search_cv.fit(xtrain, ytrain)

In [ ]:
grid_search_cv.best_estimator_

In [ ]:
grid_search_cv.best_params_

In [ ]:
results=pd.DataFrame(grid_search_cv.cv_results_)

In [ ]:
results

In [ ]:
results.sort_values(by='rank_test_score')

In [ ]:
# try writing the grid search cv for logistic regression and RandomForest